In [1]:
import pandas as pd

# 1. Load the data
df = pd.read_csv('bus_data_Line1.csv')

# 2. Data Cleaning & Type Conversion
df['Logged_At_Timestamp'] = pd.to_datetime(df['Logged_At_Timestamp'])
df['Next_Stop'] = df['Next_Stop'].astype('Int64')

# NEW CONDITION: Drop rows where "Next_Stop" is NULL (NaN)
df = df.dropna(subset=['Next_Stop'])

# 3. Sort by timestamp
df = df.sort_values(by=['Logged_At_Timestamp'])

# 4. Create your final snapshot grouped dataframe
final_stop_snapshots = df.groupby(['Trip_ID', 'Current_Stop']).last().reset_index()

# 5. Save whichever file version you need
final_stop_snapshots.to_csv('final_bus_snapshots.csv', index=False, encoding='utf-8-sig')

In [1]:
# Combine ALL csv files

import glob
import os
import pandas as pd

# 1. Define the directory path
folder_path = "/Users/chanhsuanhung/Downloads/Bus data"

# 2. Grab all CSV files in that specific folder
# This matches any file ending with .csv in your Bus data directory
file_pattern = os.path.join(folder_path, "*.csv")
all_files = glob.glob(file_pattern)

# 3. Read and combine all CSV files into a single DataFrame
# Using a list comprehension is the fastest way to do this
df_list = [pd.read_csv(file) for file in all_files]
combined_df = pd.concat(df_list, ignore_index=True)

# 4. (Optional) Save the combined data to a new CSV file
output_path = os.path.join(folder_path, "combined_bus_data.csv")
combined_df.to_csv(output_path, index=False)

print(f"Successfully combined {len(all_files)} files!")
print(f"Saved to: {output_path}")

Successfully combined 10 files!
Saved to: /Users/chanhsuanhung/Downloads/Bus data/combined_bus_data.csv


In [3]:
import pandas as pd

# Load the datasets
df_bus = pd.read_csv("combined_bus_data.csv")
df_weather = pd.read_csv("weather.csv")

# 1. Convert time columns to datetime format
df_bus["Logged_At_Timestamp"] = pd.to_datetime(df_bus["Logged_At_Timestamp"])
df_weather["time"] = pd.to_datetime(df_weather["time"])

# 2. Create a new column in bus data that rounds down to the nearest hour
df_bus["weather_lookup_time"] = df_bus["Logged_At_Timestamp"].dt.floor("h")

# 3. Merge the datasets on the hourly keys
df_merged = pd.merge(
    df_bus, df_weather, left_on="weather_lookup_time", right_on="time", how="left"
)

# 4. Display the dataframes
print("--- Weather Dataframe Preview ---")
display(df_weather.head())

print("\n--- Merged Dataframe Preview ---")
display(df_merged.head())

--- Weather Dataframe Preview ---


,time,temperature_2m (°C),apparent_temperature (°C),relative_humidity_2m (%),rain (mm),precipitation (mm),surface_pressure (hPa),wind_speed_10m (km/h),wind_speed_100m (km/h),soil_temperature_0_to_7cm (°C),soil_temperature_7_to_28cm (°C),soil_temperature_28_to_100cm (°C)
0,2026-06-29 00:00:00,22.8,23.9,77,0.0,0.0,1013.2,13.6,23.1,25.2,28.1,19.4
1,2026-06-29 01:00:00,21.9,23.2,74,0.0,0.0,1013.7,8.8,17.7,24.4,27.9,19.4
2,2026-06-29 02:00:00,21.6,23.5,76,0.0,0.0,1013.3,4.4,12.2,24.4,27.9,19.4
3,2026-06-29 03:00:00,20.9,22.9,80,0.0,0.0,1013.1,4.0,10.7,23.6,27.7,19.4
4,2026-06-29 04:00:00,20.6,21.7,80,0.0,0.0,1013.6,9.5,18.7,23.0,27.5,19.4



--- Merged Dataframe Preview ---


,id,Logged_At_Timestamp,Line,Vehicle_ID,Trip_ID,Direction,Direction_ID,Current_Stop,Next_Stop,Delay_Min,...,apparent_temperature (°C),relative_humidity_2m (%),rain (mm),precipitation (mm),surface_pressure (hPa),wind_speed_10m (km/h),wind_speed_100m (km/h),soil_temperature_0_to_7cm (°C),soil_temperature_7_to_28cm (°C),soil_temperature_28_to_100cm (°C)
0,1,2026-07-06 04:48:26,5,2550,060726_100164_2_5_10,Nienberge Hannaschweg,1,4722802,47231.0,0.0,...,14.7,90.0,0.0,0.0,1010.7,11.8,23.3,16.5,20.4,18.7
1,2,2026-07-06 04:49:26,5,2550,060726_100164_2_5_10,Nienberge Hannaschweg,1,4722802,47231.0,0.0,...,14.7,90.0,0.0,0.0,1010.7,11.8,23.3,16.5,20.4,18.7
2,3,2026-07-06 04:50:26,5,2550,060726_100164_2_5_10,Nienberge Hannaschweg,1,4722802,47231.0,0.0,...,14.7,90.0,0.0,0.0,1010.7,11.8,23.3,16.5,20.4,18.7
3,4,2026-07-06 04:51:26,5,2359,060726_100072_2_5_9,Hiltrup Bahnhof,2,46930,46931.0,0.0,...,14.7,90.0,0.0,0.0,1010.7,11.8,23.3,16.5,20.4,18.7
4,5,2026-07-06 04:51:26,5,2550,060726_100164_2_5_10,Nienberge Hannaschweg,1,4722802,47231.0,0.0,...,14.7,90.0,0.0,0.0,1010.7,11.8,23.3,16.5,20.4,18.7


In [7]:
# 1. Extract useful time features from the timestamp
df_merged["hour"] = df_merged["Logged_At_Timestamp"].dt.hour
df_merged["day_of_week"] = df_merged["Logged_At_Timestamp"].dt.dayofweek
# Day of the Week: 0-Mon, 1-Tue, 2-Wed, 3-Thu, 4-Fri, 5-Sat, 6-Sun


# 2. Define function to group into Weekday (0), Saturday (1), and Sunday (2)
def group_day_type(day):
    if day < 5:
        return 0  # Monday to Friday
    elif day == 5:
        return 1  # Saturday
    else:
        return 2  # Sunday


# 3. Apply the function to create the new column
df_merged["day_type"] = df_merged["day_of_week"].apply(group_day_type)

# 4. Display the resulting dataframe preview
print("--- Merged Dataframe with Time Features ---")
display(df_merged.tail())

--- Merged Dataframe with Time Features ---


,id,Logged_At_Timestamp,Line,Vehicle_ID,Trip_ID,Direction,Direction_ID,Current_Stop,Next_Stop,Delay_Min,...,precipitation (mm),surface_pressure (hPa),wind_speed_10m (km/h),wind_speed_100m (km/h),soil_temperature_0_to_7cm (°C),soil_temperature_7_to_28cm (°C),soil_temperature_28_to_100cm (°C),hour,day_of_week,day_type
524327,47668,2026-07-03 21:40:23,15,5585,030726_100053_12_15_487,Kinderhaus Brüningheide,1,45310,NaN,0.5,...,0.0,1017.2,16.6,25.7,23.0,21.0,18.4,21,4,0
524328,47669,2026-07-03 21:41:24,15,5585,030726_100053_12_15_487,Kinderhaus Brüningheide,1,45310,NaN,0.5,...,0.0,1017.2,16.6,25.7,23.0,21.0,18.4,21,4,0
524329,47670,2026-07-03 21:42:23,15,5585,030726_100053_12_15_487,Kinderhaus Brüningheide,1,45310,NaN,0.5,...,0.0,1017.2,16.6,25.7,23.0,21.0,18.4,21,4,0
524330,47671,2026-07-03 21:43:23,15,5585,030726_100053_12_15_487,Kinderhaus Brüningheide,1,45310,NaN,0.5,...,0.0,1017.2,16.6,25.7,23.0,21.0,18.4,21,4,0
524331,47672,2026-07-03 21:44:23,15,5585,030726_100053_12_15_487,Kinderhaus Brüningheide,1,45310,NaN,0.5,...,0.0,1017.2,16.6,25.7,23.0,21.0,18.4,21,4,0


In [18]:
import IPython.display as ipd
import pandas as pd

# 1. Target variable (Named X per instructions)
X = df_merged["Delay_Min"]

# 2. Feature variables (Named Y_raw per instructions)
Y_raw = df_merged[
    [
        "Line",
        "Current_Stop",
        "Direction_ID",
        "temperature_2m (°C)",
        "apparent_temperature (°C)",
        "relative_humidity_2m (%)",
        "rain (mm)",
        "surface_pressure (hPa)",
        "wind_speed_10m (km/h)",
    ]
]

# 3. Convert text columns (like 'Current_Stop') into numbers so Random Forest can process them
# If 'Line' is already a number, pd.get_dummies will safely leave it alone.
Y = pd.get_dummies(Y_raw, columns=["Current_Stop", "Line"], drop_first=True)

print("Target variable (X):")
ipd.display(X.head(3))

print("\nFeature variables (Y) ready for Machine Learning:")
ipd.display(Y.head(3))

Target variable (X):


0    0.0
1    0.0
2    0.0
Name: Delay_Min, dtype: float64


Feature variables (Y) ready for Machine Learning:


,Direction_ID,temperature_2m (°C),apparent_temperature (°C),relative_humidity_2m (%),rain (mm),surface_pressure (hPa),wind_speed_10m (km/h),Current_Stop_41021,Current_Stop_41032,Current_Stop_41044,...,Current_Stop_4706101,Current_Stop_4722802,Current_Stop_4755102,Current_Stop_4765501,Line_5,Line_8,Line_10,Line_14,Line_15,Line_17
0,1,15.3,14.7,90.0,0.0,1010.7,11.8,False,False,False,...,False,True,False,False,True,False,False,False,False,False
1,1,15.3,14.7,90.0,0.0,1010.7,11.8,False,False,False,...,False,True,False,False,True,False,False,False,False,False
2,1,15.3,14.7,90.0,0.0,1010.7,11.8,False,False,False,...,False,True,False,False,True,False,False,False,False,False


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [21]:
import IPython.display as ipd
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Target variable (Named X per your instructions)
X = df_merged["Delay_Min"]

# 2. Feature variables (Named Y_raw - keeping Current_Stop as text for encoding)
Y_raw = df_merged[
    [
        "Line",
        "Current_Stop",  # Must be kept as text here so we can group by it below
        "Direction_ID",
        "hour",
        "day_type",
        "temperature_2m (°C)",
        "apparent_temperature (°C)",
        "relative_humidity_2m (%)",
        "rain (mm)",
        "surface_pressure (hPa)",
        "wind_speed_10m (km/h)",
    ]
]

# One-hot encode ONLY 'Line' if it is a text column
Y_prepared = pd.get_dummies(Y_raw, columns=["Line"], drop_first=True)

# =====================================================================
# CRITICAL FIX: Pass Y_prepared first, then X (target)
# =====================================================================
Y_train, Y_test, x_train, x_test = train_test_split(
    Y_prepared, X, test_size=0.2, random_state=42
)

# --- Explicitly .copy() to avoid SettingWithCopyWarning ---
Y_train = Y_train.copy()
Y_test = Y_test.copy()

# 3. Calculate the average delays USING ONLY the training data answers
train_data = Y_train.copy()
train_data["Delay_Min"] = x_train  # Combine features with their target answers
stop_means = train_data.groupby("Current_Stop")["Delay_Min"].mean()

# 4. Map the averages back to BOTH train and test features
Y_train["Current_Stop_Encoded"] = Y_train["Current_Stop"].map(stop_means)
Y_test["Current_Stop_Encoded"] = Y_test["Current_Stop"].map(stop_means)

# 5. Fill any missing values in the test set using the global training mean
global_mean = x_train.mean()
Y_train["Current_Stop_Encoded"] = Y_train["Current_Stop_Encoded"].fillna(
    global_mean
)
Y_test["Current_Stop_Encoded"] = Y_test["Current_Stop_Encoded"].fillna(
    global_mean
)

# 6. Safely drop the original text column from both sets
Y_train = Y_train.drop(columns=["Current_Stop"])
Y_test = Y_test.drop(columns=["Current_Stop"])

print("Train features shape:", Y_train.shape)
ipd.display(Y_train.head(3))

Train features shape: (419465, 16)


,Direction_ID,hour,day_type,temperature_2m (°C),apparent_temperature (°C),relative_humidity_2m (%),rain (mm),surface_pressure (hPa),wind_speed_10m (km/h),Line_5,Line_8,Line_10,Line_14,Line_15,Line_17,Current_Stop_Encoded
370464,2,16,0,27.9,28.4,41.0,0.0,1010.0,11.9,False,True,False,False,False,False,1.601732
466424,1,17,0,26.0,25.4,44.0,0.2,1012.5,9.9,False,False,True,False,False,False,1.105906
82500,2,15,0,20.6,19.5,65.0,0.0,1010.3,16.0,False,False,False,False,False,True,1.116611


In [23]:
from sklearn.ensemble import RandomForestRegressor

# Initialize the model
model = RandomForestRegressor(n_estimators=150, random_state=42)

# --- Ensure only numeric columns are sent to Random Forest ---
# This drops any original string/text columns (like 'Direction' if left over)
Y_train_numeric = Y_train.select_dtypes(include=["number"])

# Train the model using your custom variable names (Y_train_numeric and x_train)
model.fit(Y_train_numeric, x_train)

print("Random Forest Model successfully trained!")

Random Forest Model successfully trained!


In [24]:
from sklearn.metrics import mean_absolute_error, r2_score

# --- Ensure only numeric columns are sent to the model for prediction ---
Y_test_numeric = Y_test.select_dtypes(include=["number"])

# Make predictions on the test set using your updated variable
predictions = model.predict(Y_test_numeric)

# Calculate Evaluation Metrics using your target variable (x_test)
mae = mean_absolute_error(x_test, predictions)
r2 = r2_score(x_test, predictions)

print("--- Model Evaluation Results ---")
print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")
print(f"R-squared (R²) Score: {r2:.2f}")

--- Model Evaluation Results ---
Mean Absolute Error (MAE): 1.23 minutes
R-squared (R²) Score: 0.38


In [26]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Grab the column names of the numeric features we actually trained the model on
trained_features = Y_train.select_dtypes(include=["number"]).columns

# 2. Grab how much the model cared about each column using the correct feature index
importances = pd.Series(model.feature_importances_, index=trained_features)

# Sort them and look at the top ones
print("Feature Importances:")
print(importances.sort_values(ascending=False).head(10))

Feature Importances:
Current_Stop_Encoded         0.526853
surface_pressure (hPa)       0.090012
hour                         0.087582
Direction_ID                 0.062330
apparent_temperature (°C)    0.061960
temperature_2m (°C)          0.057850
wind_speed_10m (km/h)        0.053667
relative_humidity_2m (%)     0.043143
day_type                     0.008759
rain (mm)                    0.007843
dtype: float64


In [27]:
import joblib

# Save the trained Random Forest model
joblib.dump(model, "bus_delay_rf_model.pkl")

# Save your bus stop lookup averages (crucial for encoding new data!)
joblib.dump(stop_means, "stop_means_lookup.pkl")

print("Model and encoder successfully exported!")

Model and encoder successfully exported!
